# Analiza za 9.9.2026, colormaps za EI 2 baths in fixed point T1, T2 check z equivalent T

Za logs + saved files sta fixed_baths.py in phase_map_plots.py

In [7]:
import sys
from pathlib import Path
import tqdm.notebook as tqdm 
import numpy as np
import yaml
import jax

ROOT = Path("/home/kzeleznikar/IJS-F1/Koda/EI_baths")
CFG = ROOT / "config" / "config_phonons.yaml"


import EI.ei_unified as eu
from phase_map_plots import plot_maps, scan_maps
from fixed_baths import (
    plot_bz,
    plot_dispersion,
    plot_energy,
    plot_xi,
    solve_state,
    solve_teff,
)
from utils.cosmetics import apply_plt_style
from utils.logger import logger
from utils.plotting_utils import Plotter
import EI.ei_unified as eu
import EI.ei_jax_2 as ej
import EI.ei_phonon as pb

from utils.logger import logger, tqdm_bar
from utils.plotting_utils import Plotter, plot_cmap, format_ax, add_legend, plot_cmap, plot_phase_diag, color_map
from fixed_baths import k_path, take_path, color_line, solve_state, plot_dispersion


apply_plt_style()
pt = Plotter(show=True, close=False)

from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats("png", dpi=300)

In [2]:
with CFG.open("r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

md = cfg["model"]
eq = cfg["equilibrium"]
sc = cfg["scan"]
op = cfg["open_system"]
nk = md["nk"]
bp = md["band"]["pars"]

bd = eu.tb_2d(nk["scan"], **bp)
bd_ref = eu.tb_2d(nk["reference"], **bp)
p = eu.MFPars(**md["mean_field"])

In [3]:
s0 = eu.solve_eq(bd_ref, p, **eq["initial"], **eq["solve"])
d0 = max(abs(float(s0.d)), 1.0e-12)
m0 = float(s0.m)
tc = float(eu.critical_temperature(bd_ref, p, **eq["critical"]))

logger.info("Delta_0 = %.8f", d0)
logger.info("m_0 = %.8f", m0)
logger.info("T_c = %.8f", tc)

kriticna temp bisekcija:   0%|          | 0/100 [00:00<?, ?it/s]

[21:12:58] [INFO    ] Delta_0 = 1.00005168
[21:12:58] [INFO    ] m_0 = -0.68433834
[21:12:58] [INFO    ] T_c = 0.53523542


In [4]:
tg = sc["temperature"]

tn = np.linspace( tg["min_ratio"], tg["max_ratio"], tg["nt"])

t1a = tn * tc
t2a = tn * tc
nt = tg["nt"]

st = ej.mf_state(bd_ref, p, s0.d, s0.m)
t1n, t2n = 1.0, 0.3
t1, t2 = t1n * tc, t2n * tc

In [6]:

bc = op["bath"]
fn = getattr(pb, bc["rate"])
bp = dict(bc["pars"])

bs = (
    fn(t1, bd.k, name="bath_1", **bp),
    fn(t2, bd.k, name="bath_2", **bp),
)

# Solve the state corresponding to T1 and T2.
es = {**eq["solve"], "prog": False}
se = eu.solve_eq(
    bd, p,
    t=t2,
    d=max(abs(d0), 1.0e-8),
    m=m0,
    **es,
)

sk = {
    "mix": (0.02, 0.02, 0.02),
    "tol": 1.0e-6,
    "nmax": 100000,
    "chk": 50,
    "prog": True,
}

so = ej.solve_fixed(
    bd, p, se.n.copy(), bs,
    d=float(se.d),
    m=float(se.m),
    **sk,
)

st = ej.mf_state(
    bd, p,
    d=float(so.d),
    m=float(so.m),
)


Fixed EI JAX:   0%|          | 0/2000 [00:00<?, ?it/s]

In [8]:
mu = 0

kg = np.asarray(bd.k, dtype=float)
ip = int(np.argmin(np.linalg.norm(kg, axis=1)))

gm = np.empty(
    (2, bd.size, 2, 2),
    dtype=complex,
)

for ib, b in enumerate(tqdm(bs, desc="Coupling matrices")):
    per = 2.0 * np.pi / b.a0
    qk = (kg - kg[ip] + 0.5 * per) % per - 0.5 * per

    for ik in tqdm(range(bd.size), desc=b.name, leave=False):
        gm[ib, ik] = np.asarray(jax.device_get(pb.g_mat(st, b, ik, ip, qk[ik])))

TypeError: 'module' object is not callable

In [9]:
zb = np.abs(gm[:, :, :, mu]) ** 2
z = np.concatenate(
    (
        zb,
        np.sum(zb, axis=0, keepdims=True),
    ),
    axis=0,
)

bl = (
    rf"bath 1, $T_1/T_c={t1n:g}$",
    rf"bath 2, $T_2/T_c={t2n:g}$",
    r"$\sum_b |G^{(b)}|^2$",
)

kx = kg[:, 0].reshape(bd.shape)
ky = kg[:, 1].reshape(bd.shape)

zp = z[np.isfinite(z) & (z > 0.0)]

if zp.size == 0:
    raise RuntimeError(
        "all coupling matrices are zero; check qd, amp and the orbital matrix"
    )

vmax = float(np.max(zp))
vmin = max(float(np.min(zp)), 1.0e-8 * vmax)
norm = LogNorm(vmin=vmin, vmax=vmax)

h = 0.9
fig, ax = plt.subplots(
    3,
    2,
    sharex=True,
    sharey=True,
    figsize=(2 * 3.47412, 3 * h * 3.47412),
)

nu_labels = [r"\alpha", r"\beta"]

for ib in tqdm(range(3), desc="Bath rows"):
    for nu in tqdm(
        range(2),
        desc="Bands",
        leave=False,
    ):
        zz = z[ib, :, nu].reshape(bd.shape)
        zz = np.ma.masked_less_equal(zz, 0.0)

        im = ax[ib, nu].pcolormesh(
            kx,
            ky,
            zz,
            shading="nearest",
            cmap=cmaps.lipari,
            norm=norm,
        )
        im.set_edgecolor("face")

        ax[ib, nu].set_xlim(kx.min(), kx.max())
        ax[ib, nu].set_ylim(ky.min(), ky.max())
        ax[ib, nu].text(
            0.04,
            0.94,
            rf"{bl[ib]}, $\nu={nu_labels[nu]}$",
            transform=ax[ib, nu].transAxes,
            va="top",
            bbox=dict(
                boxstyle="round,pad=0.3",
                facecolor="white",
                edgecolor="black",
                alpha=0.8,
            ),
        )

    ax[ib, 0].set_ylabel(r"$k_y$")

for a in tqdm(ax[-1], desc="Axis labels", leave=False):
    a.set_xlabel(r"$k_x$")

cax = fig.add_axes([0.15, 0.99, 0.7, 0.01])
cbar = fig.colorbar(
    im,
    cax=cax,
    orientation="horizontal",
)
cbar.ax.xaxis.set_ticks_position("top")
cbar.ax.xaxis.set_label_position("top")
cbar.set_label(
    rf"$|G_{{\nu,{mu}}}(\mathbf{{k}},\mathbf{{p}}_0)|^2$",
    labelpad=5,
)

fig.tight_layout()

plt.savefig(
    "test.pdf",
    bbox_inches="tight",
)
plt.show()

RuntimeError: all coupling matrices are zero; check qd, amp and the orbital matrix